<a href="https://colab.research.google.com/github/jaityagi63/LLM-From-Scratch/blob/main/LLM-From-Scratch%20/Tokenizer/LLM_Tokenizer_from_scratch_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
with open('/content/the-verdict.txt','r') as f: # dataset taken from Sebastian Raschka github if you want take a look at his github you will see more things
  raw_text = f.read()

print(f'len of the whole verdict: {len(raw_text)}')
print(raw_text[:100])

len of the whole verdict: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


#### using the re library we can split the sentence into token  

In [7]:
import re
"""
 if you thinking about using the python split then
 it will split the data and remove the spaces and gives the list
 but we also want the punc and common etc as token so we use the re.split
 """
data = 'hello, mfos this, is the exmaple for you.'
result = re.split(r'(\s)', data)
print(result)

['hello,', ' ', 'mfos', ' ', 'this,', ' ', 'is', ' ', 'the', ' ', 'exmaple', ' ', 'for', ' ', 'you.']


now the question that one should ask why remove the spaces :<br>

removing the space depend on the application you are developing
1.   for some application removing spaces reduce the memory and compution requiremnet
2.   sometime kepping the spaces is neccessary
     1. when the training a model which is sensitive to the structure of the data
      - ex : python code is sensitve to indent/space




In [8]:
result = re.split(r'([,.]|\s)',data)
# Filter out empty strings from the result
result = [token for token in result if token.strip() != '']
print(result)

['hello', ',', 'mfos', 'this', ',', 'is', 'the', 'exmaple', 'for', 'you', '.']


In [9]:
data = 'hello, mfos this, is the [ ] exmaple-- for you.'
result = re.split(r'(\s|--|[.,:;?"_!(){}<>+=\[\]|])', data)
de = [token for token in result if token and token.strip() != '']
print(de)

['hello', ',', 'mfos', 'this', ',', 'is', 'the', '[', ']', 'exmaple', '--', 'for', 'you', '.']


In [10]:
# now applying the the re filter to the raw data the verdict
result = re.split(r'(\s|--|[.,:;?"_!(){}<>+=\[\]|])',raw_text)
proceesed = [token for token in result if token and token.strip() != '']
print(proceesed)

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'Riviera', '.', '(', 'Though', 'I', 'rather', 'thought', 'it', 'would', 'have', 'been', 'Rome', 'or', 'Florence', '.', ')', '"', 'The', 'height', 'of', 'his', 'glory', '"', '--', 'that', 'was', 'what', 'the', 'women', 'called', 'it', '.', 'I', 'can', 'hear', 'Mrs', '.', 'Gideon', 'Thwing', '--', 'his', 'last', 'Chicago', 'sitter', '--', 'deploring', 'his', 'unaccountable', 'abdication', '.', '"', 'Of', 'course', "it's", 'going', 'to', 'send', 'the', 'value', 'of', 'my', 'picture', "'way", 'up', ';', 'but', 'I', "don't", 'think', 'of', 'that', ',', 'Mr', '.', 'Rickh

Now that, we have token we can make the ids and to do that.<br>
we first need the vocab and then enumerte the vocab    <br>
in this vocab every token is mapped to a unique id/number

- vocab --> unqiue token in our dataset

In [11]:
vocab = sorted(set(proceesed))
len(vocab) # we have 1154 unique token in the dataset


1154

In [138]:
wtoi = {token:inte for inte,token in enumerate(vocab)}
for v in list(wtoi.items())[-10:]:
    print(v) # now every single vocab has an iq associated with it

('yourself', 1153)
('<|unk|>', 1154)
('<|endoftext|>', 1155)
('<|im_start|>', 1156)
('<|im_end|>', 1157)
('<|assistant|>', 1158)
('<|user|>', 1159)
('<|system|>', 1160)
('<|context|>', 1161)
('<|/context|>', 1162)


In [137]:
# we also need a mapping from id to token
itow = {inte:token for token,inte in wtoi.items()}
for v in list(itow.items())[-10:]:
    print(v)

(1144, 'year')
(1145, 'years')
(1146, 'yellow')
(1147, 'yet')
(1148, 'you')
(1149, "you'd")
(1150, "you're")
(1151, 'younger')
(1152, 'your')
(1153, 'yourself')


In [14]:
class SimpleTokenizer:
  def __init__(self,vocab):
    # vocab are all the unique token in the dataset
    self.str_to_int = {token:inte for inte,token in enumerate(vocab)}
    self.int_to_str = {inte:token for token,inte in self.str_to_int.items()}

  def encode(self,text):
      preprocessed_text_raw = re.split(r'(\s|--|[.,:;?"_!(){}<>+=\[\]|])', text)
      preprocessed_text = [token for token in preprocessed_text_raw if  token.strip() != '']

      ids = [self.str_to_int[token] for token in preprocessed_text]
      return ids

  def decode(self,ids):

      text = " ".join([self.int_to_str[id] for id in ids])
      # remove the spaces with corresponding specif punc
      text = re.sub(r'(\s)([,.?!"()\'])',r'\2',text)
      return text

In [15]:
tokeniser = SimpleTokenizer(vocab)

text_segment_from_processed = " ".join(proceesed[0:10])

ids = tokeniser.encode(text_segment_from_processed)
print(f"Encoded IDs: {ids}")
print(f"Number of encoded IDs: {len(ids)}")


Encoded IDs: [62, 52, 167, 1024, 70, 44, 840, 133, 274, 503]
Number of encoded IDs: 10


In [16]:
decoded_text = tokeniser.decode(ids)
print(f"Decoded text: {decoded_text}")

Decoded text: I HAD always thought Jack Gisburn rather a cheap genius


In [17]:
" ".join(proceesed[0:10])

'I HAD always thought Jack Gisburn rather a cheap genius'

In [18]:
# now the next question should be what happens if given text not in vocal
text = "yo my man you got the work to do"
ids = tokeniser.encode(text)
print(f"Encoded IDs: {ids}") # from this we can undertand that we need a very large and diverse vocab

KeyError: 'yo'

special context token: <br>

- <|unk|> --> when encountering an unknown word not in vocab  
- <|endoftext|> --> when working with multiple text source add this token  between the source
- <|im_start|> and <|im_end|> --> usually used in model like Quen
- <|assistant|> --> : Tells the model, "The following text is generated by you (the AI)."
- <|user|>: --> Tells the model, "The following text was typed by the human."
- <|system|>: --> Prepends the hidden system prompt (the initial instructions that tell the AI how to behave).
- <|context|> & <|/context|>

sometime these special token is used as fences to tell the model some specifc instruction

In [20]:
all_vocab = sorted(list(set(proceesed)))
all_vocab.extend(["<|unk|>","<|endoftext|>","<|im_start|>","<|im_end|>","<|assistant|>","<|user|>","<|system|>","<|context|>","<|/context|>"])

In [26]:
vocab = {token:integer for integer,token in enumerate(all_vocab)}
for v in list(vocab.items())[-10:]:
    print(v)

('yourself', 1153)
('<|unk|>', 1154)
('<|endoftext|>', 1155)
('<|im_start|>', 1156)
('<|im_end|>', 1157)
('<|assistant|>', 1158)
('<|user|>', 1159)
('<|system|>', 1160)
('<|context|>', 1161)
('<|/context|>', 1162)


but we will for now work with only two unk and endoftext okay😎

In [139]:
class SimpleTokenizer2:
  def __init__(self, vocab):
    self.str_to_int = vocab
    self.int_to_str = {integer:token for token,integer in vocab.items()}

  def encode(self, text):
      # Prioritize special tokens in the regex to prevent them from being split by punctuation
      preprocessed_text_raw = re.split(r'([,.:;?_!"()\']|--|\s)', text)
      preprocessed_text = [token for token in preprocessed_text_raw if token and token.strip() != '']
      preprocessed_text = [
         item if item in self.str_to_int
         else "<|unk|>" for item in preprocessed_text]

      ids = [self.str_to_int[token] for token in preprocessed_text]
      return ids

  def decode(self, ids):
      text = " ".join([self.int_to_str[id] for id in ids])
      # Clean up spaces before common punctuation marks
      text = re.sub(r'(\s)([,.?/!"()\'])', r'\2', text)
      return text

In [140]:
# Re-testing with the special tokens removed from the split regex
test_text = "I am tired <|endoftext|> hello"
ids = token.encode(test_text)
decoded = token.decode(ids)

print(f"Encoded IDs: {ids}")
print(f"Decoded text: {decoded}")

Encoded IDs: [62, 168, 1036, 1155, 1154]
Decoded text: I am tired <|endoftext|> <|unk|>


In [141]:
token = SimpleTokenizer2(vocab)
text = "yo my man you got the work to do"
ids = token.encode(text)
print(f"Encoded IDs: {ids}")

Encoded IDs: [1154, 719, 678, 1148, 520, 1007, 1139, 1037, 372]


In [142]:
# Re-instantiate with the corrected class to verify the fix
token = SimpleTokenizer2(vocab)
test_text = "Hello!<|endoftext|>"

ids = token.encode(test_text)
decoded = token.decode(ids)

print(f"Tokens: {[token.int_to_str[i] for i in ids]}")
print(f"Encoded IDs: {ids}")
print(f"Decoded text: {decoded}")

Tokens: ['<|unk|>', '!', '<|endoftext|>']
Encoded IDs: [1154, 0, 1155]
Decoded text: <|unk|>! <|endoftext|>


In [143]:
# Re-defining the text with endoftext to test the fix
text1 = "I am tired of of this shit breaking"
text2 = "hello how was your day"
text = " <|endoftext|> ".join((text1, text2))
print(f"Original text: {text}")

Original text: I am tired of of this shit breaking <|endoftext|> hello how was your day


In [145]:
# Re-running the endoftext test with the corrected class
ids = token.encode(text)
decoded_text = token.decode(ids)

In [146]:
print(f"Encoded IDs: {ids}")
print(f"Decoded text: {decoded_text}")

Encoded IDs: [62, 168, 1036, 744, 744, 1020, 1154, 246, 1155, 1154, 579, 1097, 1152, 332]
Decoded text: I am tired of of this <|unk|> breaking <|endoftext|> <|unk|> how was your day
